<a href="https://colab.research.google.com/github/KCCalder/CSC502-XGBoost-Split-Finding/blob/main/BenchMarkingWPySpark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import time
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml.feature import Bucketizer
import numpy as np

In [ ]:
# Algorithm 1: Exact Greedy Split
def pyspark_exact_split(df, feature_col, g_col, h_col, reg_lambda=1.0):
    """
    Algorithm 1: Exact greedy split finding. From XGBoost paper
    Evaluates gain at every possible split
    """

    # Total gradient and hessian. statistics for the entire node, before splitting
    totals = df.agg(
        F.sum(g_col).alias("total_G"),  # Sum of gradients over all samples
        F.sum(h_col).alias("total_H")   # Sum of Hessians over all samples
    ).collect()[0]  # Result to driver (single row)

    total_G, total_H = totals["total_G"], totals["total_H"]

    # Score of the node before splitting
    base_score = (total_G**2) / (total_H + reg_lambda)

    # Global sort by the feature. Sum up to the current row
    window_spec = Window.orderBy(feature_col) \
                        .rowsBetween(Window.unboundedPreceding, Window.currentRow)

    # Prefix sums, for each row i:
    df_cumulatives = df.withColumn(
        "G_L", F.sum(g_col).over(window_spec) # sum of gradients for all rows <= i
    ).withColumn(
        "H_L", F.sum(h_col).over(window_spec) # sum of Hessians for all rows <= i
    )

    # Right-side statistics
    df_scores = df_cumulatives.withColumn(
        "G_R", F.lit(total_G) - F.col("G_L")
    ).withColumn(
        "H_R", F.lit(total_H) - F.col("H_L")
    )

    # Split gain at each row
    df_scores = df_scores.withColumn(
        "score",
        (F.col("G_L")**2 / (F.col("H_L") + reg_lambda)) +
        (F.col("G_R")**2 / (F.col("H_R") + reg_lambda)) -
        base_score
    )

    # Find best split
    best_row = df_scores.orderBy(F.col("score").desc()) \
                        .limit(1) \
                        .collect()[0]

    # best split, corresponding gain
    return best_row[feature_col], best_row["score"]

In [ ]:
# Helper functions for approximately optimal split finding algorithms

def get_node_stats(df, g_col, h_col, reg_lambda, extra_aggs=None):
    """
    Computes global statistics for the current node.

    """

    # Total gradient and Hessian
    aggs = [F.sum(g_col).alias("G"), F.sum(h_col).alias("H")]
    # extra_aggs is for pyspark_fast_approximate_split
    if extra_aggs:
        aggs.extend(extra_aggs)

    stats = df.agg(*aggs).collect()[0]

    # Handling missing values
    total_G = stats["G"] if stats["G"] else 0.0
    total_H = stats["H"] if stats["H"] else 0.0

    # base, a.k.a. parent node score
    base_score = (total_G**2) / (total_H + reg_lambda)

    return total_G, total_H, base_score, stats


def build_histogram(binned_df, g_col, h_col):
    """
    Builds a histogram over buckets
    Map: assign rows to buckets
    Reduce: aggregate statistics per bucket
    """

    return [row.asDict() for row in binned_df.groupBy("bucket").agg(
        F.sum(g_col).alias("G_bucket"),
        F.sum(h_col).alias("H_bucket")
    ).orderBy("bucket").collect()]


def evaluate_sparsity_aware_splits(histogram, total_G, total_H, base_score, reg_lambda, split_mapper):
    """
    Evaluate splits using histogram bins instead of raw data
    Handles missing values by treating them as a unit and trying them out on
    both the laft and right side.
    """

    best_score, best_split_val, best_direction = 0.0, None, None

    # Pass 1: Missing values assigned to right
    G_L, H_L = 0.0, 0.0

    for i in range(len(histogram) - 1):

        # Left-side stats
        G_L += histogram[i]['G_bucket']
        H_L += histogram[i]['H_bucket']

        # Right side implicitly includes remaining buckets, and all missing values
        G_R, H_R = total_G - G_L, total_H - H_L

        # Gain
        score = (G_L**2 / (H_L + reg_lambda)) + \
                (G_R**2 / (H_R + reg_lambda)) - base_score

        # Update best split
        if score > best_score:
            best_score, best_direction = score, "Right"
            best_split_val = split_mapper(histogram[i]['bucket'])

    # pass 2: Missing values assigned to LEFT
    G_R, H_R = 0.0, 0.0

    for i in range(len(histogram) - 1, 0, -1):

        # Right-side stats
        G_R += histogram[i]['G_bucket']
        H_R += histogram[i]['H_bucket']

        # Left side implicitly includes remaining buckets, and all missing values
        G_L, H_L = total_G - G_R, total_H - H_R

        score = (G_L**2 / (H_L + reg_lambda)) + \
                (G_R**2 / (H_R + reg_lambda)) - base_score

        if score > best_score:
            best_score, best_direction = score, "Left"
            best_split_val = split_mapper(histogram[i-1]['bucket'])

    return best_split_val, best_score, best_direction


In [ ]:
# Algorithm 2: Weighted Quantile Sketch (Custom MapReduce)

def get_local_summary(partition_iterator, num_bins=100):
    """
    Worker node function. Local weighted summary for one partition.
    """
    data = list(partition_iterator)
    if not data:
        return []

    x = np.array([row[0] for row in data])
    h = np.array([row[1] for row in data])

    # Local sort
    idx = np.argsort(x)
    x, h = x[idx], h[idx]

    total_h = np.sum(h)
    cum_h = np.cumsum(h)

    # Sample points at evenly spaced weight intervals
    thresholds = np.linspace(0, total_h, num_bins)
    summary = []

    for t in thresholds:
        pos = np.searchsorted(cum_h, t)
        if pos < len(x):
            summary.append((float(x[pos]), float(h[pos])))

    return summary


def fast_weighted_sketch(df, feature_col, h_col, eps=0.1):
    """
    Driver-level function: Distributed merge & prune phase.
    Mimics the Weighted Quantile Sketch from the XGBoost paper.
    """
    # Map: Each partition computes a local summary
    local_summaries_rdd = df.select(feature_col, h_col).rdd \
                            .mapPartitions(lambda part: get_local_summary(part))

    # Reduce: Collect summaries back to the driver
    all_summaries = local_summaries_rdd.collect()

    # Combine all local summaries into a global summary
    all_summaries.sort(key=lambda x: x[0])

    x_merged = np.array([s[0] for s in all_summaries])
    h_merged = np.array([s[1] for s in all_summaries])

    total_h = np.sum(h_merged)
    cum_h = np.cumsum(h_merged)

    # Global split candidates based on epsilon
    num_buckets = int(1.0 / eps)
    target_ranks = [i * eps * total_h for i in range(1, num_buckets)]

    final_splits = []
    for target in target_ranks:
        pos = np.searchsorted(cum_h, target)
        if pos < len(x_merged):
            final_splits.append(x_merged[pos])

    unique_splits = sorted(list(set(final_splits)))

    # Tnfinity boundaries so Bucketizer doesn't crash
    return [-float('inf')] + unique_splits + [float('inf')]

def pyspark_approximate_split_WQS(df, feature_col, g_col, h_col, eps=0.1, reg_lambda=1.0):
    """
    Custom Weighted Quantile Sketch. Closest to XGBoost algorithmically.
    Slow in Spark due to Python overhead, Breaks JVM optimization pipeline
    """

    # Base stats
    total_G, total_H, base_score, _ = get_node_stats(df, g_col, h_col, reg_lambda)

    # Weighted sketch
    valid_df = df.filter(F.col(feature_col).isNotNull())

    splits = fast_weighted_sketch(valid_df, feature_col, h_col, eps)

    if len(splits) <= 2:
        return None, 0.0, None

    # Histogram
    binned_df = Bucketizer(
        splits=splits,
        inputCol=feature_col,
        outputCol="bucket"
    ).transform(valid_df)

    histogram = build_histogram(binned_df, g_col, h_col)

    # Evaluate Splits
    split_mapper = lambda b: splits[int(b) + 1]

    return evaluate_sparsity_aware_splits(
        histogram, total_G, total_H, base_score, reg_lambda, split_mapper
    )


# Algorithm 3: Unweight Quantile Sketch (approxQuantile)
def pyspark_approximate_split(df, feature_col, g_col, h_col, eps=0.1, reg_lambda=1.0):
    """
    Uses Spark's built-in approxQuantile. Not Hessian-weighted, unlike XGBoost
    """

    # Base stats
    total_G, total_H, base_score, _ = get_node_stats(df, g_col, h_col, reg_lambda)

    # Approximate quantiles
    valid_df = df.filter(F.col(feature_col).isNotNull())

    probabilities = [i * eps for i in range(1, int(1.0 / eps))]

    raw_splits = valid_df.approxQuantile(
        feature_col,
        probabilities,
        relativeError=0.01
    )

    # Split boundaries
    splits = [-float('inf')] + sorted(list(set(raw_splits))) + [float('inf')]

    if len(splits) <= 2:
        return None, 0.0, None

    # Histogram
    binned_df = Bucketizer(
        splits=splits,
        inputCol=feature_col,
        outputCol="bucket"
    ).transform(valid_df).cache()

    histogram = build_histogram(binned_df, g_col, h_col)

    # Evaluate splits
    split_mapper = lambda b: splits[int(b) + 1]

    res = evaluate_sparsity_aware_splits(histogram, total_G, total_H, base_score, reg_lambda, split_mapper)

    binned_df.unpersist()

    return res

# Algorithm 4: Fast Math Binning (F.floor)
def pyspark_fast_approximate_split(df, feature_col, g_col, h_col, num_buckets=32, reg_lambda=1.0):
    """
    Uniform binning. No sorting
    """

    # totals + feature bounds. One pass
    extra_aggs = [F.min(feature_col).alias("min"), F.max(feature_col).alias("max")]
    total_G, total_H, base_score, stats = get_node_stats(df, g_col, h_col, reg_lambda, extra_aggs)

    min_val, max_val = stats["min"], stats["max"]

    bin_width = (max_val - min_val + 1e-9) / num_buckets

    # Assign rows to buckets
    binned_df = df.filter(F.col(feature_col).isNotNull()).withColumn(
        "bucket", F.floor((F.col(feature_col) - F.lit(min_val)) / F.lit(bin_width))
    ).cache()

    # Histogram
    histogram = build_histogram(binned_df, g_col, h_col)

    # Evaluate splits
    split_mapper = lambda b: min_val + (b + 1) * bin_width # bucket index to actual feature value

    res = evaluate_sparsity_aware_splits(histogram, total_G, total_H, base_score, reg_lambda, split_mapper)

    # Clean cache
    binned_df.unpersist()

    return res



In [ ]:
# Benchmarking
spark = SparkSession.builder \
    .appName("ComprehensiveSplitComparison") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

sample_sizes = [100_000, 200_000, 500_000, 1_000_000, 1_500_000, 2_000_000, 2_500_000, 5_000_000, 7_500_000, 10_000_000]
for i in range(len(sample_sizes)):
  N_SAMPLES = sample_sizes[i]
  print(f"Generating JVM-native DataFrame with {N_SAMPLES} rows...")
  print("Injecting a true split pattern at X = 2.0...\n")

  df = spark.range(N_SAMPLES).select(
      (F.randn(seed=42) * 5).alias("feature_x")
  ).withColumn(
      "g",
      F.when(F.col("feature_x") > 2.0, F.randn(seed=43) * 0.5 - 1.0)
        .otherwise(F.randn(seed=43) * 0.5 + 1.0)
  ).withColumn(
      "h",
      F.rand(seed=44) * 0.9 + 0.1
  ).cache()

  df.count()

  results = {}

  print("="*60)
  print("1. RUNNING ALGORITHM 1: EXACT GREEDY (Global Sort)")
  print("="*60)
  t0 = time.time()

  try:
      val_ex, score_ex, dir_ex = pyspark_exact_split(df, "feature_x", "g", "h")
  except ValueError:
      val_ex, score_ex = pyspark_exact_split(df, "feature_x", "g", "h")
      dir_ex = "N/A"

  t_ex = time.time() - t0
  results['Exact Greedy'] = (t_ex, val_ex, score_ex, dir_ex)
  print(f"Time: {t_ex:.4f}s | Split: {val_ex:.4f} | Score: {score_ex:.4f}\n")

  print("="*60)
  print("2. RUNNING ALGORITHM 2: TRUE WQS (Custom MapReduce)")
  print("="*60)
  t0 = time.time()
  val_wqs, score_wqs, dir_wqs = pyspark_approximate_split_WQS(df, "feature_x", "g", "h", eps=0.03125) # 1/32 = 0.03125
  t_wqs = time.time() - t0
  results['True WQS'] = (t_wqs, val_wqs, score_wqs, dir_wqs)
  print(f"Time: {t_wqs:.4f}s | Split: {val_wqs:.4f} | Score: {score_wqs:.4f}\n")

  print("="*60)
  print("3. RUNNING ALGORITHM 3: NATIVE UNWEIGHTED (approxQuantile)")
  print("="*60)
  t0 = time.time()
  val_aq, score_aq, dir_aq = pyspark_approximate_split(df, "feature_x", "g", "h", eps=0.03125)
  t_aq = time.time() - t0
  results['Native approxQuantile'] = (t_aq, val_aq, score_aq, dir_aq)
  print(f"Time: {t_aq:.4f}s | Split: {val_aq:.4f} | Score: {score_aq:.4f}\n")

  print("="*60)
  print("4. RUNNING ALGORITHM 4: FAST MATH BINNING (F.floor)")
  print("="*60)
  t0 = time.time()

  val_fast, score_fast, dir_fast = pyspark_fast_approximate_split(df, "feature_x", "g", "h", num_buckets=32)
  t_fast = time.time() - t0
  results['Fast Math Binning'] = (t_fast, val_fast, score_fast, dir_fast)
  print(f"Time: {t_fast:.4f}s | Split: {val_fast:.4f} | Score: {score_fast:.4f}\n")


  print("="*75)
  print("FINAL PERFORMANCE BENCHMARK SUMMARY")
  print("="*75)
  print(f"{'Algorithm':<25} | {'Time (s)':<10} | {'Split Val':<10} | {'Score':<15} | {'Missing Dir'}")
  print("-" * 75)
  for name, (t, v, s, d) in results.items():
      v_str = f"{v:.4f}" if v is not None else "None"
      s_str = f"{s:.4f}" if s is not None else "0.0000"
      print(f"{name:<25} | {t:<10.4f} | {v_str:<10} | {s_str:<15} | {d}")

Generating JVM-native DataFrame with 100000 rows...
Injecting a true split pattern at X = 2.0...

1. RUNNING ALGORITHM 1: EXACT GREEDY (Global Sort)
Time: 3.3152s | Split: 1.9999 | Score: 164215.8559

2. RUNNING ALGORITHM 2: TRUE WQS (Custom MapReduce)
Time: 6.1370s | Split: 1.9711 | Score: 162673.2161

3. RUNNING ALGORITHM 3: NATIVE UNWEIGHTED (approxQuantile)
Time: 2.5360s | Split: 1.9329 | Score: 160539.8772

4. RUNNING ALGORITHM 4: FAST MATH BINNING (F.floor)
Time: 1.7049s | Split: 1.5133 | Score: 139593.0001

FINAL PERFORMANCE BENCHMARK SUMMARY
Algorithm                 | Time (s)   | Split Val  | Score           | Missing Dir
---------------------------------------------------------------------------
Exact Greedy              | 3.3152     | 1.9999     | 164215.8559     | N/A
True WQS                  | 6.1370     | 1.9711     | 162673.2161     | Left
Native approxQuantile     | 2.5360     | 1.9329     | 160539.8772     | Left
Fast Math Binning         | 1.7049     | 1.5133     | 

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/socket.py", line 720, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 